# SAMPIC Noise Filtering & Average Shape

Use this notebook to identify and remove purely noisy waveforms from SAMPIC data, inspect summary statistics, and visualise the aggregate shape of the remaining signals.
Baseline and peak are waveform estimates, not removed vendor scalar fields. Per-channel summaries use unique IDs `64*FEB + channel` to avoid merging boards. Thresholds are illustrative and use the displayed units.


## Workflow

1. Load the unpacked ROOT file and required libraries.
2. Classify each hit waveform using configurable noise rejection criteria.
3. Report statistics (overall and per channel) on filtered vs retained hits.
4. Visualise channel-level filtering and build a 2D heatmap plus average waveform for the surviving signals.

Tune the thresholds in the **Filtering Parameters** cell to match your dataset.

In [ ]:
# Edit this path, then run the notebook from the top.
# Absolute paths and ~ are supported; relative paths start at the app directory.
DATA_FILE = ".artifacts/bank_format_verification/run00266/output.root"


In [ ]:
from pathlib import Path
import sys, os
# Works when Jupyter starts in the repository or a notebook subdirectory.
APP_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / 'notebooks/notebook_support.py').is_file())
if str(APP_ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(APP_ROOT / 'notebooks'))
from notebook_support import (open_data, present, hit_waveform, hit_baseline,
    hit_amplitude, hit_peak, waveform_entries as find_waveform_entries,
    sampic_unit, integer, waveform_packets, sampic_event_bytes)
import ROOT
r = ROOT
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
root_file, tree, DATA_FILE = open_data('SAMPIC', DATA_FILE)
f = file = root_file
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.25})
UNIT = sampic_unit(tree)
print('Waveform display units:', UNIT)


## Filtering Parameters

Adjust these thresholds to control how aggressively noise is removed. By default we treat a waveform as noise when:

- Its dynamic range (max - min) stays below `AMP_THRESHOLD` **and** the baseline-subtracted waveform maximum is small.
- Its sample-to-sample variation is tiny (`STD_THRESHOLD`).
- Its time-over-threshold metric is essentially zero (`TOT_THRESHOLD`).
- Waveforms dropping below `-NEGATIVE_DIP_THRESHOLD` (after baseline subtraction) are flagged for inspection but kept in the retained set.

All surviving waveforms are baseline-subtracted before building the heatmap/averages.

`Time-over-Threshold (TOT)` is the duration (in ns) the signal stays above a hardware threshold. Some datasets leave this at zero, so the notebook only uses the TOT cut when a positive value is present.

In [ ]:
# Display-unit thresholds: mV for calibrated files, ADC counts otherwise.
AMP_THRESHOLD = 40.0 if UNIT == 'mV' else 400.0
PEAK_THRESHOLD = 160.0 if UNIT == 'mV' else 1600.0
STD_THRESHOLD = 0.2 if UNIT == 'mV' else 2.0
NEGATIVE_DIP_THRESHOLD = 20.0 if UNIT == 'mV' else 200.0
TOT_THRESHOLD = None  # ns; disabled unless explicitly set. Zero TOT is not a measurement.
MAX_EVENTS = None
BASELINE_SAMPLES = 8
print('Threshold units:', UNIT, '(illustrative defaults; tune to your detector)')


In [ ]:
def extract_waveform(hit):
    """Return waveform samples as a numpy array of floats."""
    return hit_waveform(hit)


def classify_hit(hit):
    waveform = extract_waveform(hit)
    if waveform.size == 0:
        return waveform, {
            'amplitude': 0.0,
            'peak': 0.0,
            'tot': getattr(hit, 'tot_value', 0.0),
            'std': 0.0,
            'min_value': 0.0,
            'has_negative_dip': False,
            'is_noise': True,
        }

    baseline = waveform[:BASELINE_SAMPLES].mean() if waveform.size >= BASELINE_SAMPLES else waveform.mean()
    centered = waveform - baseline
    amplitude = float(centered.max() - centered.min())
    peak = float(centered.max())
    tot_raw = getattr(hit, 'tot_value', None)
    tot = float(tot_raw) if tot_raw is not None else float('nan')
    std = float(centered.std())
    min_value = float(centered.min())

    core_noise = (amplitude < AMP_THRESHOLD and abs(peak) < PEAK_THRESHOLD and std < STD_THRESHOLD)
    tot_measured = not np.isnan(tot) and tot > 0.0
    if tot_measured and TOT_THRESHOLD is not None:
        noise_like = core_noise and tot <= TOT_THRESHOLD
    else:
        noise_like = core_noise

    has_negative_dip = min_value < -NEGATIVE_DIP_THRESHOLD

    return centered, {
        'amplitude': amplitude,
        'peak': peak,
        'tot': tot if not np.isnan(tot) else None,
        'std': std,
        'min_value': min_value,
        'has_negative_dip': has_negative_dip,
        'is_noise': noise_like,
    }


In [ ]:
summary = {
    'events_scanned': 0,
    'total_hits': 0,
    'noise_hits': 0,
    'negative_hits': 0,
}
channel_stats = defaultdict(lambda: {'total': 0, 'noise': 0, 'negative': 0})
non_noise_waveforms = []
non_noise_channels = []
non_noise_amplitudes = []
non_noise_tot = []
negative_waveforms = []
negative_info = []

entries = tree.GetEntries() if MAX_EVENTS is None else min(MAX_EVENTS, tree.GetEntries())

for idx in range(entries):
    tree.GetEntry(idx)
    summary['events_scanned'] += 1
    if tree.GetBranch('has_sampic_event') and not bool(tree.has_sampic_event):
        continue
    event = tree.sampic_event
    for hit_idx, hit in enumerate(event.hits):
        waveform, stats = classify_hit(hit)
        ch = 64 * int(hit.fe_board_index) + int(hit.channel)  # unique FEB/channel ID
        summary['total_hits'] += 1
        channel_stats[ch]['total'] += 1

        if stats['is_noise']:
            summary['noise_hits'] += 1
            channel_stats[ch]['noise'] += 1
            continue

        non_noise_waveforms.append(waveform)
        non_noise_channels.append(ch)
        non_noise_amplitudes.append(stats['amplitude'])
        non_noise_tot.append(stats['tot'])

        if stats['has_negative_dip']:
            summary['negative_hits'] += 1
            channel_stats[ch]['negative'] += 1
            negative_waveforms.append(waveform)
            negative_info.append({
                'event': idx,
                'hit': hit_idx,
                'channel': ch,
                'min_value': stats['min_value'],
                'amplitude': stats['amplitude'],
                'peak': stats['peak'],
            })

print(f"Processed {summary['events_scanned']} events")
print(f"Total hits inspected: {summary['total_hits']}")
retained = summary['total_hits'] - summary['noise_hits']
print(f"Retained hits: {retained}")
print(f"Noise-like hits filtered: {summary['noise_hits']} ({summary['noise_hits']/max(summary['total_hits'],1):.2%})")
if retained:
    print(f"Hits with baseline undershoot (< -{NEGATIVE_DIP_THRESHOLD} {UNIT}): {summary['negative_hits']} ({summary['negative_hits']/retained:.2%})")
else:
    print("Hits with baseline undershoot: n/a (no retained hits)")


In [ ]:
channels = sorted(channel_stats.keys())
print(f"Channels seen: {channels}")
print('Per-channel summary:')
print(f"{'Ch':>3} | {'Hits':>6} | {'Filtered':>8} | {'Retained':>8} | {'Under BL':>9} | {'Under %':>8}")
print('-' * 70)
for ch in channels:
    totals = channel_stats[ch]
    filtered = totals['noise']
    retained = totals['total'] - filtered
    negative = totals['negative']
    frac = (filtered / totals['total']) if totals['total'] else 0.0
    neg_frac = (negative / retained) if retained else 0.0
    print(f"{ch:>3d} | {totals['total']:>6d} | {filtered:>8d} | {retained:>8d} | {negative:>9d} | {neg_frac:>7.1%}")


In [ ]:
if channels:
    filtered_counts = [channel_stats[ch]['noise'] for ch in channels]
    retained_counts = [channel_stats[ch]['total'] - channel_stats[ch]['noise'] for ch in channels]
    negative_counts = [channel_stats[ch]['negative'] for ch in channels]

    x = np.arange(len(channels))
    width = 0.3

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - width, retained_counts, width, label='Retained', color='tab:blue')
    ax.bar(x, filtered_counts, width, label='Filtered (noise)', color='tab:orange')
    ax.bar(x + width, negative_counts, width, label='Retained w/ undershoot', color='tab:red')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{ch//64}/{ch%64}' for ch in channels])
    ax.set_xlabel('FEB/channel')
    ax.set_ylabel('Hit count')
    ax.set_title('Channel-wise hit breakdown')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No channels found – check filtering thresholds or input file.')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

if non_noise_waveforms:
    lengths = [len(wf) for wf in non_noise_waveforms]
    min_len = min(lengths)
    if min_len == 0:
        raise RuntimeError("Encountered zero-length waveform; cannot build heatmap.")

    # --- Baseline subtraction ---
    baseline_samples = BASELINE_SAMPLES  # use first N samples as baseline reference
    processed = []
    for wf in non_noise_waveforms:
        wf = wf[:min_len]
        wf = wf - np.mean(wf[:baseline_samples])  # baseline subtract
        processed.append(wf)
    trimmed = np.stack(processed)


    # --- Flatten for histogram ---
    sample_indices = np.tile(np.arange(min_len), trimmed.shape[0])
    values = trimmed.reshape(-1)

    v_min, v_max = np.percentile(values, [0.5, 99.5])
    bins = [min_len, 200]
    hist, xedges, yedges = np.histogram2d(
        sample_indices, values, bins=bins, range=[[0, min_len], [v_min, v_max]]
    )

    # --- Heatmap ---
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(
        hist.T,
        origin="lower",
        aspect="auto",
        extent=[0, min_len, yedges[0], yedges[-1]],
        cmap="magma",
        norm=LogNorm(vmin=1, vmax=max(hist.max(), 2))
    )
    fig.colorbar(im, ax=ax, label="Counts per bin")
    ax.set_xlabel("Sample index")
    ax.set_ylabel(f"Amplitude [{UNIT}] (baseline-subtracted)")
    ax.set_title("Heatmap of retained waveform shapes")

    # --- Mean waveform (same scale as heatmap) ---
    mean_wave = trimmed.mean(axis=0)
    std_wave = trimmed.std(axis=0)
    ax.plot(np.arange(min_len), mean_wave, color="cyan", lw=2, label="Mean waveform")
    ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

    # --- Average waveform ±1σ envelope ---
    fig, ax = plt.subplots(figsize=(14, 4))
    x = np.arange(min_len)
    ax.plot(x, mean_wave, color="tab:blue", label="Mean waveform")
    ax.fill_between(x, mean_wave - std_wave, mean_wave + std_wave,
                    color="tab:blue", alpha=0.25, label="±1σ envelope")
    ax.axhline(0, color="k", lw=0.8, ls="--")
    ax.set_xlabel("Sample index")
    ax.set_ylabel(f"Amplitude [{UNIT}] (baseline-subtracted)")
    ax.set_title("Average retained waveform (baseline-subtracted, normalized)")
    ax.legend()
    plt.tight_layout()
    plt.show()

else:
    print("No non-noise waveforms collected – relax thresholds or check data file.")


## Baseline undershoot inspection

In [ ]:
if summary['negative_hits']:
    print(f"Waveforms flagged with baseline undershoot (< -{NEGATIVE_DIP_THRESHOLD} {UNIT}): {summary['negative_hits']}")
    preview_count = min(10, summary['negative_hits'])
    print(f"Showing first {preview_count} entries (event, hit, channel, min {UNIT}, peak, amplitude):")
    for entry in negative_info[:preview_count]:
        print(f"  event {entry['event']}, hit {entry['hit']}, ch {entry['channel']} | "
              f"min {entry['min_value']:.1f}, peak {entry['peak']:.1f}, amp {entry['amplitude']:.1f}")
else:
    print('No baseline undershoots detected with current threshold.')


In [ ]:
def plot_negative_examples(max_examples=4):
    if not negative_waveforms:
        print('No negative-dip waveforms to plot.')
        return
    n = min(max_examples, len(negative_waveforms))
    fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=True)
    if n == 1:
        axes = [axes]
    for ax, wf, meta in zip(axes, negative_waveforms[:n], negative_info[:n]):
        ax.plot(wf, color='tab:red')
        ax.axhline(0, color='k', linestyle='--', linewidth=0.8)
        ax.set_ylabel(UNIT)
        ax.set_title(f"Event {meta['event']} | Hit {meta['hit']} | Ch {meta['channel']} | min {meta['min_value']:.1f} {UNIT}")
    axes[-1].set_xlabel('Sample index')
    plt.tight_layout()
    plt.show()

plot_negative_examples()


In [ ]:
if negative_waveforms:
    lengths = [len(wf) for wf in negative_waveforms]
    min_len = min(lengths)
    trimmed = np.stack([wf[:min_len] for wf in negative_waveforms])

    sample_indices = np.tile(np.arange(min_len), trimmed.shape[0])
    values = trimmed.reshape(-1)
    v_min, v_max = np.percentile(values, [0.5, 99.5])
    bins = [min_len, 200]
    hist, xedges, yedges = np.histogram2d(
        sample_indices, values, bins=bins, range=[[0, min_len], [v_min, v_max]]
    )

    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(
        hist.T,
        origin='lower',
        aspect='auto',
        extent=[0, min_len, yedges[0], yedges[-1]],
        cmap='coolwarm'
    )
    fig.colorbar(im, ax=ax, label='Counts per bin')
    ax.set_xlabel('Sample index')
    ax.set_ylabel(f'Amplitude [{UNIT}] (baseline-subtracted)')
    ax.set_title('Heatmap of undershooting waveforms')
    plt.tight_layout()
    plt.show()
else:
    print('No undershooting waveforms available for heatmap.')


# All Channel heatmaps

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# --- Sanity check ---
if len(non_noise_waveforms) == 0 or len(non_noise_channels) == 0:
    print("No non-noise waveform data available.")
else:
    # --- Grouping setup ---
    non_noise_channels = np.array(non_noise_channels)
    unique_channels = np.unique(non_noise_channels)
    n_channels = len(unique_channels)
    print(f"Plotting {n_channels} channels with retained waveforms...")

    # --- Fixed waveform parameters ---
    target_len = min(map(len, non_noise_waveforms))  # never invent padded samples
    baseline_samples = BASELINE_SAMPLES
    y_min, y_max = (-102.4, 102.4) if UNIT == "mV" else (-1024, 1024)

    # --- Standardize waveform length and baseline ---
    processed = []
    for wf in non_noise_waveforms:
        wf = np.asarray(wf, dtype=float)
        if len(wf) < target_len:
            wf = np.pad(wf, (0, target_len - len(wf)), constant_values=wf[-1] if len(wf) > 0 else 0)
        elif len(wf) > target_len:
            wf = wf[:target_len]
        wf = wf - np.mean(wf[:baseline_samples])
        processed.append(wf)
    processed = np.stack(processed)
    y_min, y_max = float(processed.min()), float(processed.max())
    if y_min == y_max: y_min, y_max = y_min-1, y_max+1

    # --- Figure 1: per-channel heatmaps (8×8 grid) ---
    fig, axes = plt.subplots(n_channels, 1, figsize=(12, 3*n_channels), squeeze=False, constrained_layout=True)
    for ax, ch in zip(axes.flat, unique_channels):
        if ch not in unique_channels:
            ax.axis("off")
            continue

        mask = (non_noise_channels == ch)
        wfs = processed[mask]
        if wfs.size == 0:
            ax.axis("off")
            continue

        sample_indices = np.tile(np.arange(target_len), wfs.shape[0])
        values = wfs.reshape(-1)

        bins = [target_len, 200]
        hist, xedges, yedges = np.histogram2d(
            sample_indices, values, bins=bins, range=[[0, target_len], [y_min, y_max]]
        )

        im = ax.imshow(
            hist.T,
            origin="lower",
            aspect="auto",
            extent=[0, target_len, y_min, y_max],
            cmap="magma",
            norm=LogNorm(vmin=1, vmax=max(hist.max(), 2))
        )
        ax.set_title(f"FEB {ch//64}, channel {ch%64}")
        ax.set_xlim(0, target_len)
        ax.set_ylim(y_min, y_max)
        ax.set_xlabel("Sample index")
        ax.set_ylabel(f"Baseline-subtracted [{UNIT}]")

    fig.suptitle("Per-channel waveform heatmaps (baseline-subtracted)", fontsize=18)
    plt.show()

    # --- Figure 2: per-channel mean ±1σ envelopes (8×8 grid) ---
    fig, axes = plt.subplots(n_channels, 1, figsize=(12, 3*n_channels), squeeze=False, constrained_layout=True)
    x = np.arange(target_len)
    for ax, ch in zip(axes.flat, unique_channels):
        if ch not in unique_channels:
            ax.axis("off")
            continue

        mask = (non_noise_channels == ch)
        wfs = processed[mask]
        if wfs.size == 0:
            ax.axis("off")
            continue

        mean_wave = wfs.mean(axis=0)
        std_wave = wfs.std(axis=0)
        ax.plot(x, mean_wave, color="tab:blue", lw=1)
        ax.fill_between(x, mean_wave - std_wave, mean_wave + std_wave,
                        color="tab:blue", alpha=0.3)
        ax.axhline(0, color="k", lw=0.5, ls="--")
        ax.set_xlim(0, target_len)
        ax.set_ylim(y_min, y_max)
        ax.set_title(f"FEB {ch//64}, channel {ch%64}")
        ax.set_xlabel("Sample index")
        ax.set_ylabel(f"Baseline-subtracted [{UNIT}]")

    fig.suptitle("Per-channel average waveforms (baseline-subtracted)", fontsize=18)
    plt.show()
